# RECON 3D Reconstruction Engine
This notebook provides a free GPU-powered photogrammetry backend for the RECON app.

In [ ]:
# CELL 1: Install system dependencies
!apt-get update
!apt-get install -y colmap
!pip install pycolmap flask flask-cors pyngrok open3d trimesh Pillow numpy

In [ ]:
# CELL 2: Mount Google Drive and create working folder
from google.colab import drive
import os

drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/3DScanPOC/'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
print(f"Google Drive mounted. Output folder ready at: {OUTPUT_DIR}")

In [ ]:
# CELL 3: Define the COLMAP pipeline
import subprocess
import shutil
import time
import glob
import open3d as o3d
import trimesh
from PIL import Image

def run_colmap_pipeline(image_dir):
    # 1. Validation
    images = glob.glob(os.path.join(image_dir, "*.jpg")) + glob.glob(os.path.join(image_dir, "*.jpeg"))
    if len(images) < 10:
        raise ValueError(f"Not enough images ({len(images)}). Need at least 10.")

    # 2. Setup Workspace
    workspace = "/content/colmap_workspace"
    if os.path.exists(workspace): shutil.rmtree(workspace)
    os.makedirs(os.path.join(workspace, "sparse"), exist_ok=True)
    os.makedirs(os.path.join(workspace, "dense"), exist_ok=True)

    # 3. Feature Extraction
    print("Running Feature Extraction...")
    subprocess.run([
        "colmap", "feature_extractor",
        "--database_path", os.path.join(workspace, "database.db"),
        "--image_path", image_dir,
        "--ImageReader.camera_model", "SIMPLE_RADIAL",
        "--ImageReader.single_camera", "1"
    ], check=True)

    # 4. Exhaustive Matching
    print("Running Matching...")
    subprocess.run([
        "colmap", "exhaustive_matcher",
        "--database_path", os.path.join(workspace, "database.db")
    ], check=True)

    # 5. Incremental Mapping
    print("Running Sparse Mapping...")
    subprocess.run([
        "colmap", "mapper",
        "--database_path", os.path.join(workspace, "database.db"),
        "--image_path", image_dir,
        "--output_path", os.path.join(workspace, "sparse")
    ], check=True)

    # Check for sparse output
    sparse_model_dir = os.path.join(workspace, "sparse/0")
    if not os.path.exists(sparse_model_dir) or not os.listdir(sparse_model_dir):
        raise ValueError("Reconstruction failed - 0 images registered")

    # 6. Dense Reconstruction
    print("Running Dense Reconstruction (GPU enabled)... ")
    subprocess.run([
        "colmap", "image_undistorter",
        "--image_path", image_dir,
        "--input_path", sparse_model_dir,
        "--output_path", os.path.join(workspace, "dense"),
        "--output_type", "COLMAP"
    ], check=True)

    subprocess.run([
        "colmap", "patch_match_stereo",
        "--workspace_path", os.path.join(workspace, "dense"),
        "--PatchMatchStereo.geom_consistency", "true"
    ], check=True)

    subprocess.run([
        "colmap", "stereo_fusion",
        "--workspace_path", os.path.join(workspace, "dense"),
        "--output_path", os.path.join(workspace, "fused.ply")
    ], check=True)

    # 7. Mesh Conversion (Poisson)
    print("Generating Mesh...")
    pcd = o3d.io.read_point_cloud(os.path.join(workspace, "fused.ply"))
    pcd.estimate_normals()
    
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
    
    # Trim low density
    import numpy as np
    vertices_to_remove = densities < np.quantile(densities, 0.01)
    mesh.remove_vertices_by_mask(vertices_to_remove)

    timestamp = int(time.time())
    obj_path = os.path.join(OUTPUT_DIR, f"{timestamp}_model.obj")
    o3d.io.write_triangle_mesh(obj_path, mesh)
    
    return obj_path

In [ ]:
# CELL 4: Flask server + ngrok tunnel
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import torch
import threading

# --- CONFIG START ---
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"
# --- CONFIG END ---

app = Flask(__name__)
CORS(app)

@app.route('/health', methods=['GET'])
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU Mode)"
    return jsonify({"status": "online", "gpu": gpu_name})

@app.route('/process', methods=['POST'])
def process():
    input_dir = "/content/scan_input/"
    if os.path.exists(input_dir): shutil.rmtree(input_dir)
    os.makedirs(input_dir)

    files = request.files.getlist("images")
    for file in files:
        file.save(os.path.join(input_dir, file.filename))
    
    try:
        obj_path = run_colmap_pipeline(input_dir)
        return jsonify({"status": "complete", "obj_path": obj_path})
    except ValueError as e:
        return jsonify({"status": "failed", "error": str(e)}), 400
    except Exception as e:
        return jsonify({"status": "failed", "error": f"Unexpected error: {str(e)}"}), 500

def run_app():
    app.run(port=5000)

if __name__ == "__main__":
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(5000).public_url
    
    print("\n================================================")
    print("COLAB IS READY")
    print("Copy this URL into your .env file as COLAB_NGROK_URL:")
    print(public_url)
    print("================================================\n")
    
    threading.Thread(target=run_app).start()